## 贝叶斯公式推导
- 假设男生概率60%, 女生概率40%
- 男生只会穿长裤,女生一半穿长裤,一半穿裙子
- 如果对面走来一个穿着裤子的人,那她是女生的概率是多少?

```
假设总人数是U
P(长裤|男) 代表男生的时候穿长裤的概率
P(长裤|女) 代表女生的时候穿长裤的概率
穿长裤的男生 = U*P(男)*P(长裤|男)
穿长裤的女生 = U*P(女)*P(长裤|女)
则穿长裤的总人数 = U*P(男)*P(长裤|男) + U*P(女)*P(长裤|女)
P(女|长裤) = U*P(女)*P(长裤|女) / (U*P(男)*P(长裤|男) + U*P(女)*P(长裤|女))
          = P(女) * P(长裤|女) / (P(男)*P(长裤|男) + P(女)*P(长裤|女))
          = P(女) * P(长裤|女) / P(长裤)
P(A|B) = P(A) *P(B|A) / P(B)
```

----




In [19]:
def getDataSet(dataPath=r"./SMSSpamCollection"):
    with open(dataPath, encoding='utf-8') as f:
        txt_data = f.readlines()
    # 所有邮件
    data = []
    # 标签
    classTag = []
    # 垃圾邮件
    spam_data_num = 0
    # 正常邮件
    ham_data_num = 0
    for line in txt_data:
        line_split = line.strip("\n").split('\t')
        if line_split[0] == "spam":
            data.append(line_split[1])
            spam_data_num += 1
            classTag.append(1)
        elif line_split[0] == "ham":
            data.append(line_split[1])
            ham_data_num += 1
            classTag.append(0)
    print("数据集大小为{}, 其中垃圾邮件数量为{}，正常邮件数量为{}".format(len(data), spam_data_num, ham_data_num))
    return data, classTag

In [20]:
doclist, labels = getDataSet()

数据集大小为5574, 其中垃圾邮件数量为747，正常邮件数量为4827


In [21]:
import numpy as np
import re

def createVocablist(doclist):
    vocabSet = set([])
    for docContent in doclist:
        document = data_preprocess(docContent)
        vocabSet = vocabSet | set(document)
    return {word: i for i, word in enumerate(list(vocabSet))}

def data_preprocess(txt_content):
    # 将输入转换为小写并将特殊字符替换为空格
    temp_info = re.sub(r'\W', ' ', txt_content.lower())
    # 根据空格将其分割为一个一个单词
    words = re.split(r'\s+', temp_info)
    # 返回长度大于等于3的所有单词
    return list(filter(lambda x: len(x) >= 3, words))

def setOfWord2Vec(vocablist, inputSet):
    returnVec = [0] * len(vocablist)
    for word in inputSet:
        if word in vocablist:
            returnVec[vocablist[word]] += 1
    return returnVec

def trainNB(trainMat, trainLabel):
    numTrainDocs = len(trainMat)
    numWords = len(trainMat[0])
    p1 = sum(trainLabel) / float(numTrainDocs)
    p0Num = np.ones(numWords)
    p1Num = np.ones(numWords)
    p0Denom = 2
    p1Denom = 2
    for i in range(numTrainDocs):
        if trainLabel[i] == 1:
            p1Num += trainMat[i]
            p1Denom += sum(trainMat[i])
        else:
            p0Num += trainMat[i]
            p0Denom += sum(trainMat[i])

    p1Vec = np.log(p1Num / p1Denom)
    p0Vec = np.log(p0Num / p0Denom)
    return p0Vec, p1Vec, p1

def classifyNB(wordVec, p0Vec, p1Vec, p1_class):
    p1 = np.log(p1_class) + sum(wordVec * p1Vec)
    p0 = np.log(1 - p1_class) + sum(wordVec * p0Vec)
    if p1 > p0:
        return 1
    else:
        return 0

def spam(doclist, classTag):
    vocablist = createVocablist(doclist)
    trainMat = []
    trainLabel = []
    for docIndex in range(len(doclist)):
        words = data_preprocess(doclist[docIndex])  # 先做分词
        trainMat.append(setOfWord2Vec(vocablist, words))
        trainLabel.append(classTag[docIndex])

    p0Vec, p1Vec, p1_class = trainNB(np.array(trainMat), np.array(trainLabel))

    errorCount = 0
    for docIndex in range(len(doclist)):
        wordVec = setOfWord2Vec(vocablist, data_preprocess(doclist[docIndex]))
        pred = classifyNB(np.array(wordVec), np.array(p0Vec), np.array(p1Vec), p1_class)
        if pred != classTag[docIndex]:
            errorCount += 1

    print("acc:", 1 - errorCount/len(doclist))

In [22]:
doclist, labels = getDataSet()
spam(doclist, labels)

数据集大小为5574, 其中垃圾邮件数量为747，正常邮件数量为4827
acc: 0.9840330104054539


In [32]:
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import CountVectorizer

text = ["hbj is a good boy, and good father",
        "tx also is a good gril, she is very beautiful, she is very smart",
        "we are good, we have a good family"
        ]
cv = CountVectorizer()
x = cv.fit_transform(text)
x, cv.get_feature_names_out(), x.toarray()


(<Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 20 stored elements and shape (3, 17)>,
 array(['also', 'and', 'are', 'beautiful', 'boy', 'family', 'father',
        'good', 'gril', 'have', 'hbj', 'is', 'she', 'smart', 'tx', 'very',
        'we'], dtype=object),
 array([[0, 1, 0, 0, 1, 0, 1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 3, 2, 1, 1, 2, 0],
        [0, 0, 1, 0, 0, 1, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 2]]))

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 20 stored elements and shape (3, 17)>